# 论文基本信息

| 项目        | 内容                                                                                    |
| :-------- | :------------------------------------------------------------------------------------ |
| **标题**    | Do Not Let Low-Probability Tokens Over-Dominate in RL for LLMs                        |
| **作者**    | Zhihe Yang, Xufang Luo, Zilong Wang, Dongqi Han, Zhiyuan He, Dongsheng Li, Yunjian Xu |
| **机构**    | The Chinese University of Hong Kong; Microsoft Research Asia                          |
| **arXiv** | [2505.12929]https://arxiv.org/abs/2505.12929                                        |
| **发表时间**  | 2025年5月                                                                               |
| **代码**    | [github.com/zhyang2226/AR-Lopti](https://github.com/zhyang2226/AR-Lopti)              |


# 研究背景与核心问题
## RLVR 中的更新偏差
RLVR（如 GRPO）已成为提升 LLM 推理能力的核心技术。
- 已有研究关注了<font color='red'>多种训练偏差来源</font>
  - **如奖励黑客**
  - **长度偏差**、
  - **KL 散度约束**等
- <font color='red'>但论文指出还存在一个被严重忽视但至关重要的偏差</font>：
  - "we identify another important source of update bias in RL training, which is orthogonal to aforementioned ones and has rarely been noted in prior research. This bias arises from the gradient perspective and is strongly correlated with the token probabilities." 
  
## 核心现象
在 GRPO 训练过程中，按概率四分位数将 token 分组后观察：
- 低概率 token（概率 < 0.25）：<font color='red'>梯度范数极大</font>
- 高概率 token（概率 > 0.75）：梯度范数几乎被压制
  - "low-probability tokens generate disproportionately larger gradients compared to high-probability ones." 
  
**更惊人的是干预实验：**
- 仅更新低概率 token 时，高概率 token 的概率变化模式与全量更新几乎一致；
- <font color='red'>而仅更新高概率 token 时，模式完全不同。这说明低概率 token 的梯度"劫持"了整个更新过程。</font>

# 理论解释
## 核心定理
论文通过理论分析揭示了**梯度范数**与 **token 概率**<font color='red'>之间的定量关系：</font>
  - "for an LLM comprising a benign neural network, the gradient norm of any intermediate activation corresponding to a single token is bounded between two values proportional to (1−π)/π , where π  is the token's probability." 
这意味着：

| Token 概率 π | 梯度范数上界 ∝ (1-π)/π | 含义            |
| :------------: | :--------------------------: | :------------ |
|    0.01（极低）    |              ~99             | <font color='red'>**梯度巨大**，更新主导</font> |
|     0.25（低）    |               3              | 梯度显著          |
|     0.75（高）    |             0.33             | 梯度被压制         |
|    0.99（极高）    |             ~0.01            | **梯度几乎为零**    |

这与之前讨论的 $∇logπ=(1/π)⋅∇π$  机制一致：
- <font color='red'>概率越低，对数概率对参数越敏感，梯度范数越大</font>。

# 危害验证：高概率 token 的学习被干扰

## 更新方向正确率
论文统计了"正优势 token（应该被强化）在更新后概率是否真正增加"的比例：
  - "the proportion of correct update directions for tokens with probability greater than 0.75 is even slightly less than 50%." 
这意味着：
- <font color='red'>对于高概率 token，RL 更新有一半以上的概率是在错误方向上调整</font>
- <font color='blue'>它们本应是模型最"确定"的部分，却因为梯度被低概率 token 淹没而无法被正确优化</font>

## 为什么高概率 token 很重要？
  - "masking high-probability tokens leads to a significant degradation in the performance of the baseline GRPO. Therefore, reducing the influence of low-probability tokens on high-probability ones holds great potential for advancing RL training." 
  
**虽然高概率 token 的梯度小，但它们是模型推理骨架（如关键数学符号、逻辑连接词）的载体。如果它们不能被精确校准，模型的整体推理质量会下降。**

# 提出的两种方法
## 方法 1：Advantage Reweighting（优势重加权）
**核心思想**：根据 token 概率直接缩小低概率 token 的优势权重。
$\hat{A}_{i,t} = \left[ \alpha \cdot \pi_\theta(o_{i,t}) + (1 - \alpha) \right] \cdot \hat{A}_{i,t}^{\text{old}}$

| 参数                    | 作用                                                           |
| :-------------------- | :----------------------------------------------------------- |
| α ∈ [0,1]    | 重加权强度超参                                                      |
| $\pi_\theta(o_{i,t})$ | **当前策略对该 token 的概率**                                             |
| 系数范围                  | 当 **$\pi \to 0$** 时系数 $\to (1-\alpha)$;当 **$\pi \to 1$** 时系数 $\to 1$ |


**效果**：<font color='red'>低概率 token 的优势被线性压缩，高概率 token 基本保持原权重。实现简单，几乎无额外计算开销。</font>

## 方法 2：Lopti（Low-Probability Token Isolation）
**核心思想**：将低概率 token 和高概率 token **隔离更新**，<font color='blue'>避免低概率梯度淹没高概率梯度。</font>

**算法流程**：

In [ ]:
1. 设定概率阈值 η（如 0.5）
2. 将 batch 中的 token 分为两组：
   - 低概率组：π_old ≤ η
   - 高概率组：π_old > η
3. 第一轮更新：只更新低概率组（高概率组 mask 掉）
4. 第二轮更新：只更新高概率组（低概率组 mask 掉）

**关键细节**：<font color='red'>顺序不能反转。必须先低后高：</font>
  - "The intuition behind Lopti...If a positive high-probability token is affected in the correct direction (i.e., its probability increases), its gradient becomes smaller in the subsequent stage...Conversely, if its probability decreases, its gradient will dominate within the high-probability token group." 
**如果先更新高概率组，训练会在第 4 个 epoch 后崩溃。**



# 实验结果
## 主实验：K&K Logic Puzzle
在 Qwen2.5-3B-Instruct 上用 GRPO 训练：

| 方法                      | 相对 GRPO 基线提升 |
| :---------------------- | :----------: |
| GRPO Baseline           |       —      |
| + Advantage Reweighting |  **+35.9%**  |
| + Lopti                 |  **+38.5%**  |
| + AR + Lopti（联合）        |  **+46.2%**  |

**消融实验**

| 实验                  | 结论                      |
| :------------------ | :---------------------- |
| <font color='red'>屏蔽高概率 token </font>        | **性能显著下降 → 高概率 token 不可忽略** |
| 反转 Lopti 更新顺序（<font color='red'>先高后低</font>） | <font color='blue'>训练崩溃 → 顺序至关重要  </font>         |
| $\alpha$ 超参扫描       | 存在稳定的最优区间               |
| $\eta$ 超参扫描         | $\eta=0.5$ 是通用有效设置      |


# 与相关工作的关系
## 与论文3：《On the Direction of RLVR Updates》的关系
### 那篇论文发现：
- <font color='red'>"RLVR **真正的学习发生在概率极低的区域**，这些 token 从 Base 到 RL 的概率增幅最大。"</font>

本文的发现看似矛盾，实则互补：

|          | 《On the Direction...》 | 本文                       |
| :------- | :-------------------- | :----------------------- |
| **发现**   | 低概率 token 的 Δlog p 最大 | <font color='red'>低概率 token 的梯度范数过大，过度主导更新 </font>|
| **评价**   | **低概率 token 是"金矿"**       | **低概率 token 是"噪声源"**         |
| **解决方案** | **放大**低概率 token 的优势   | **抑制**低概率 token 的优势      |


### 统一视角：
  - <font color='red'>低概率 token 确实承载了重要的推理信号（论文2）</font>
  - **但它们数量庞大、梯度过强，如果不加控制，会淹没高概率 token 的精细调整（本文）**
  - <font color='blue'>最优策略可能是选择性关注低概率 token，而非让它们"统治"整个梯度场</font>
  
### 与《Does RL Really Incentivize...》的关系
- 那篇论文认为 RLVR 只是"从 Base 已有策略中选择更高分的"。
- <font color='red'>本文从另一个角度补充</font>：
  - **即使在这个"选择"过程中，RL 的更新机制也存在结构性偏差——低概率 token 的过度主导导致选择过程不够精确。**


# 核心贡献
- 问题识别：
  - <font color='red'>首次系统揭示了 RL 训练中低概率 token 因梯度范数过大而过度主导更新的现象</font>
- 理论解释：
  - <font color='blue'>证明了梯度范数与 token 概率之间的 (1−π)/π  定量关系</font>
- 实证验证：
  - 通过干预实验和方向正确率统计，量化了这种主导对高概率 token 学习的损害
- 实用方法：
  - 提出了几乎零开销的 Advantage Reweighting 和有效的 Lopti，在多个基准上显著提升 GRPO 性能

# 局限
- 计算成本：Lopti 需要两次前向/反向传播，计算开销翻倍
- 超参数依赖：<font color='blue'>α  和 η  需要针对任务调优</font>
- 适用范围：主要在可验证奖励任务（数学、逻辑谜题）上验证，开放式生成任务的效果待探索

# 一句话总结
## 这篇论文的核心洞见是：
- RL 训练中低概率 token 的梯度"太大了"，大到淹没了高概率 token 的学习信号。
   - 这不是"要不要学低概率 token"的问题，而是"不能让它们一家独大"的问题。
   - 通过简单的概率感知重加权或隔离更新，可以让高、低概率 token 各得其所，显著提升 RL 训练的效率和最终性能。